# Project 3, Part A: Exploring the Power of Flow Models — Student Notebook

Use the current Part A assignment webpage for equations and deliverables.
Complete the student functions, run the provided display cells, and present your results
on your project webpage. This notebook follows the same three sections:

1. Visualizing Training and Sampling
2. Conditional Generation and Guidance
3. Creative Applications

## Start here
1. Select a **GPU**: Colab **Runtime → Change runtime type**, or Kaggle **Settings → Accelerator**. Enable Internet on Kaggle.
2. Run **Setup** and upload `parta_runtime_assets.zip` when prompted.
3. Complete the exercises below and run their cells.

Binary support and images stay in one external ZIP so Colab never formats them as source code.
Use a fresh runtime when switching between Part A and Part B. Downloads are automatically
reused within the session. Save your notebook with outputs before disconnecting.

In [ ]:
#@title Setup (run once)
# Large binary assets intentionally live outside the notebook. This prevents
# Colab's editor / "better printing" pass from parsing megabytes of base64.
import os, sys, time, io, zipfile
from pathlib import Path

_cs180_started = time.perf_counter()
IN_COLAB = 'google.colab' in sys.modules
IN_KAGGLE = Path('/kaggle/working').is_dir()
PART = 'parta'
_work = Path('/content/cs180-hw3') if IN_COLAB else (Path('/kaggle/working') if IN_KAGGLE else Path.cwd())
_work.mkdir(parents=True, exist_ok=True)
_archive_name = 'parta_runtime_assets.zip'
_archive_path = Path.cwd() / _archive_name

if not (_work / 'flow_model.py').is_file():
    if not _archive_path.is_file() and IN_COLAB:
        from google.colab import files
        print(f'Upload {_archive_name} (one file; no source code will be expanded).')
        _uploaded = files.upload()
        if _archive_name not in _uploaded:
            raise FileNotFoundError(f'Expected {_archive_name}; received {list(_uploaded)}')
        _archive_path = Path.cwd() / _archive_name
    if not _archive_path.is_file():
        raise FileNotFoundError(
            f'Place {_archive_name} next to this notebook, then rerun Setup.')
    with zipfile.ZipFile(_archive_path) as _archive:
        for _entry in _archive.infolist():
            _target = (_work / _entry.filename).resolve()
            if _work.resolve() not in _target.parents:
                raise ValueError('Unsafe runtime archive path')
        _archive.extractall(_work)

os.chdir(_work)
if str(_work) not in sys.path:
    sys.path.insert(0, str(_work))
from colab_setup import configure
configure(PART, install_dependencies=IN_COLAB or IN_KAGGLE)
print('Setup complete. Continue with the notebook below.')


In [ ]:
# Project-specific inputs come from parta_runtime_assets.zip.
from pathlib import Path
_input_dir = Path.cwd() / 'inputs_512'
_expected_inputs = ['01_aether_cliffs.png', '02_twilight_castle.jpg', '03_deep_dark_warden.jpg', '06_aether_sun_spirit.png', '07_handdrawn_twilight_castle.png', '08_handdrawn_ice_dragon.png', '09_aether_garden.png', '10_cataclysm_ender_golem.png']
_missing_inputs = [_name for _name in _expected_inputs if not (_input_dir / _name).is_file()]
if _missing_inputs:
    raise FileNotFoundError(f'Missing runtime inputs: {_missing_inputs}')
print('Found', len(_expected_inputs), 'project inputs in', _input_dir)


In [ ]:
import os, json, time, math
from pathlib import Path
from uuid import uuid4
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display
from torchvision.transforms.functional import gaussian_blur
from flow_model import FlowModel, preprocess, to_pil, MODEL_ID, MODEL_REVISION, TEXT_REVISION

torch.set_num_threads(4)
ROOT = Path.cwd()
GROUP = os.environ.get("FLOW_GROUP", "all")
if GROUP not in ("all", "core", "editing", "illusions"):
    raise ValueError("FLOW_GROUP must be all, core, editing, or illusions.")
# A new folder for this run keeps earlier results available.
RUN_NAME = time.strftime("%Y%m%d-%H%M%S") + "-" + uuid4().hex[:6]
OUT = ROOT / "results" / "parta_updated" / RUN_NAME
OUT.mkdir(parents=True, exist_ok=False)
SEED, H, W = 180, 0.02, 7
torch.manual_seed(SEED)
_model_key = (MODEL_REVISION, TEXT_REVISION, 'cuda:0', 'float32')
if globals().get('_loaded_model_key') != _model_key or 'model' not in globals():
    model = FlowModel()
    _loaded_model_key = _model_key
else:
    print('Reusing the already-loaded PixNerd model in this kernel.')
predict_velocity = model.predict_velocity
device = model.device
condition = model.encode("a high quality photo")
null_condition = model.encode("")


def load_input(path):
    if not str(path).strip():
        raise ValueError("Set the image path to a file you uploaded or saved in this runtime.")
    path = Path(path).expanduser()
    if not path.is_file():
        raise FileNotFoundError(f"Image not found: {path}. Check the path in the file browser.")
    return preprocess(path).to(device)


def noise(seed):
    return torch.randn((1, 3, 512, 512), generator=torch.Generator().manual_seed(seed)).to(device)


def panel(name, images, columns=4, title=None):
    items = list(images.items())
    fig, axs = plt.subplots(math.ceil(len(items) / columns), columns,
                            figsize=(4 * columns, 4 * math.ceil(len(items) / columns)),
                            squeeze=False, layout="constrained")
    if title is not None:
        fig.suptitle(title, wrap=True)
    for ax in axs.flat:
        ax.axis("off")
    for ax, (label, value) in zip(axs.flat, items):
        ax.imshow(to_pil(value))
        ax.set_title(label, fontsize=9, wrap=True)
    fig.savefig(OUT / (name + ".png"), dpi=110)
    plt.show()
    plt.close(fig)


def save(name, x):
    to_pil(x).save(OUT / (name + ".png"))


def guided(x, t, c=condition, w=W):
    return cfg_velocity(x, t, c, w, predict_velocity, null_condition)


metrics = {"group": GROUP, "seed": SEED, "step_size": H, "cfg_w": W,
           "model": MODEL_ID, "revision": MODEL_REVISION, "text_revision": TEXT_REVISION,
           "gpu": torch.cuda.get_device_name(0), "visible_devices": os.environ.get("CUDA_VISIBLE_DEVICES"),
           "torch": torch.__version__}
print("Results folder:", OUT)
print(metrics)

### Functions to implement

Run the cell below to define the function stubs. Fill in each function as you reach its
section, then rerun this cell after editing it. Follow the equations on the Part A webpage.
Image loading, text encoding, model evaluation, rotation, and low-pass filtering are provided.
The velocity-visualization exercise has its own student function in Section 1.4.

In [ ]:
# STUDENT IMPLEMENTATION: all required Part A algorithms.

def interpolate(image, noise, t):
    if not 0 <= float(t) <= 1:
        raise ValueError("t must lie in [0,1]")
    return (1-float(t))*noise + float(t)*image


@torch.inference_mode()
def one_step_denoise(x, t, condition, predict_velocity):
    return x + (1-float(t))*predict_velocity(x,t,condition)


@torch.inference_mode()
def euler_sample(x_start, t_start, step_size, velocity):
    if not 0 <= float(t_start) <= 1 or step_size <= 0:
        raise ValueError("Invalid Euler interval")
    x=x_start.detach().float().clone(); t=float(t_start); update=0
    states=[(t,x.detach().clone())]
    while t < 1-1e-10:
        h=min(float(step_size),1-t)
        x=x+h*velocity(x,t)
        t=min(1.0,t+h); update+=1
        if update%5==0 or t>=1-1e-10:
            states.append((1.0 if t>=1-1e-10 else t,x.detach().clone()))
    return x,states


@torch.inference_mode()
def cfg_velocity(x,t,condition,w,predict_velocity,null_condition=''):
    v_c=predict_velocity(x,t,condition)
    if float(w)==0: return v_c
    v_u=predict_velocity(x,t,null_condition)
    return v_c+float(w)*(v_c-v_u)


@torch.inference_mode()
def edit(image,noise,t_start,condition,predict_velocity,w=7,step_size=0.02):
    x=interpolate(image,noise,t_start)
    return euler_sample(x,t_start,step_size,
        lambda state,t: cfg_velocity(state,t,condition,w,predict_velocity,null_condition))[0]


def rotate180(x):
    return torch.flip(x,(-2,-1))


@torch.inference_mode()
def visual_anagrams(noise,condition_a,condition_b,predict_velocity,w=7,step_size=0.02):
    def velocity(x,t):
        v1=cfg_velocity(x,t,condition_a,w,predict_velocity,null_condition)
        v2=rotate180(cfg_velocity(rotate180(x),t,condition_b,w,predict_velocity,null_condition))
        return (v1+v2)/2
    return euler_sample(noise,0,step_size,velocity)[0]


def lowpass(x,sigma=16.0):
    kernel=2*round(16.5*x.shape[-1]/64)+1
    return gaussian_blur(x,[kernel,kernel],[sigma,sigma])


@torch.inference_mode()
def make_hybrids(noise,condition_a,condition_b,predict_velocity,w=7,step_size=0.02):
    def velocity(x,t):
        v1=cfg_velocity(x,t,condition_a,w,predict_velocity,null_condition)
        v2=cfg_velocity(x,t,condition_b,w,predict_velocity,null_condition)
        return lowpass(v1)+(v2-lowpass(v2))
    return euler_sample(noise,0,step_size,velocity)[0]


# 1. Visualizing Training and Sampling

Use the linear flow-matching convention from the webpage:
$x_t=(1-t)\epsilon+t x_1$, where $\epsilon\sim\mathcal{N}(0,I)$.
**Time 0 is noise; time 1 is a clean image.** The path velocity target is $x_1-\epsilon$.
We use the pretrained model; no model training is required in Part A.

### Choose your image for Sections 1.1–1.5

Use **one image of your own, not the provided image of the Campanile**, throughout these five exercises.
Upload or add your image to the runtime, then set `YOUR_IMAGE_PATH` below to its path.
In Colab, upload through the Files sidebar and copy the uploaded file's path.
In Kaggle, add the image as an input and copy its path from the file browser.

The supplied preprocessing center-crops and resizes the image to $512\times512$ and
normalizes RGB to $[-1,1]$. The Campanile will be loaded separately for Sections 3.1–3.2.

In [ ]:
YOUR_IMAGE_PATH = "inputs_512/01_aether_cliffs.png"
image = load_input(YOUR_IMAGE_PATH)
metrics["own_image_path"] = str(Path(YOUR_IMAGE_PATH).resolve())
display(to_pil(image))


## 1.1 Constructing Noisy Training Inputs

Implement `interpolate(image, noise, t)` using $x_t=(1-t)\epsilon+t x_1$.
Sample one noise tensor and reuse it at $t=0.25,0.50,0.75$.
Show your original image and the three noisy inputs, labeled by time.

**Hints:** Check both endpoints. Keep `image`, `epsilon`, and `noisy` for Sections 1.2–1.5;
the original image/noise pair is needed to compute the ground-truth velocity in Section 1.4.

In [ ]:
epsilon = noise(SEED)
noisy = {t: interpolate(image, epsilon, t) for t in [.25,.50,.75]}
assert torch.equal(interpolate(image,epsilon,0),epsilon)
assert torch.equal(interpolate(image,epsilon,1),image)
if GROUP in ("all","core"):
    torch.save({"image": image.cpu(), "epsilon": epsilon.cpu(),
                "noisy": {t: x.cpu() for t, x in noisy.items()}}, OUT / "noisy_inputs.pt")
    panel("training_inputs", {"original": image, **{f"t={t}":x for t,x in noisy.items()}})

## 1.2 Classical Denoising

For each saved input, restore the clean-image amplitude with x/t, then apply a Gaussian filter.
Try different blur strengths and show your best result beside each noisy input.
The provided cell sweeps sigma=1,2,4,8,16 and selects lowest mean squared error to the known clean image;
this is a baseline comparison, not a learned model or a guarantee of perceptual quality.

In [ ]:
if GROUP in ("all","core"):
    blurred = {}
    for t,x in noisy.items():
        candidates = {s: gaussian_blur(x/t, [int(6*s+1)]*2, [s,s]) for s in [1.,2.,4.,8.,16.]}
        sigma = min(candidates, key=lambda s: ((candidates[s]-image)**2).mean().item())
        blurred[t] = candidates[sigma]
        panel(f"blur_{t}", {"noisy":x,f"blur sigma={sigma}":blurred[t]}, columns=2)

## 1.3 One-Step Denoising

Implement `one_step_denoise`: x̂₁=xₜ+(1−t)vθ(xₜ,t,c).
Use exactly one prediction for each of the same three noisy tensors, with “a high quality photo”.
Display original, input, and estimate. Check velocity shape, dtype, and device.

In [ ]:
if GROUP in ("all","core"):
    v = predict_velocity(noisy[.5], .5, condition)
    assert v.shape == image.shape and v.dtype == image.dtype and v.device == image.device
    assert torch.isfinite(v).all()
    one_step = {t:one_step_denoise(x,t,condition,predict_velocity) for t,x in noisy.items()}
    for t,x in noisy.items():
        panel(f"one_step_{t}", {"original":image,"noisy":x,"one step":one_step[t]}, columns=3)

## 1.4 Visualizing Velocity

In Section 1.3, we added $(1-t)\hat v_t$ to the noisy image to estimate the clean image.
Here, visualize what the model adds and compare it with the update $x_1-x_t$
that would recover your original image.

Use the same image and saved noisy inputs at $t=0.25,0.50,0.75$, with
`"a high quality photo"`.

**Deliverables**
- Show the original image once.
- At each time, show the noisy input, predicted update $(1-t)\hat v_t$,
  ground-truth update $(1-t)v_{\mathrm{gt}}$, and an error heatmap, labeled by time.
  You may arrange the displays however you like.

**Implementation hints**
- Predict $\hat v_t=v_\theta(x_t,t,c)$. Using the saved noise, compute
  $v_{\mathrm{gt}}=x_1-\epsilon$, so $(1-t)v_{\mathrm{gt}}=x_1-x_t$.
- For the error map, use RMS across RGB **at each pixel**:
  $E_t=\sqrt{\operatorname{mean}_{\mathrm{RGB}}([(1-t)(\hat v_t-v_{\mathrm{gt}})]^2)}$.
  Square the channel differences, average over RGB, and take the square root before display scaling.
- Use one shared symmetric RGB scale for all updates, with zero at middle gray.
- Use `matplotlib.pyplot.imshow` with a linear scale for errors. Choose `vmin` and `vmax`
  that work for your image, keeping them the same at all three times.

Complete the calculation function below. The following cell supplies an example display.

In [ ]:
@torch.inference_mode()
def velocity_diagnostics(x_t,t,clean_image,epsilon,condition,predict_velocity):
    predicted_velocity=predict_velocity(x_t,t,condition)
    ground_truth_velocity=clean_image-epsilon
    predicted_update=(1-float(t))*predicted_velocity
    target_update=(1-float(t))*ground_truth_velocity
    error_map=((predicted_update-target_update).square().mean(dim=1)).sqrt()
    return predicted_update,target_update,error_map


In [ ]:
# PROVIDED display helpers. Adjust the error limits or layout for your results.
ERROR_VMIN = 0.0
ERROR_VMAX = None  # None uses the maximum across all three maps; you may choose a smaller limit.


def signed_rgb(update, limit):
    # Map the shared signed range [-limit, limit] to RGB [0, 1].
    value = update.detach().float().cpu()[0].permute(1, 2, 0)
    return (0.5 + value / (2 * limit)).numpy()


if GROUP in ("all", "core"):
    velocity_results = {}
    for t in (0.25, 0.50, 0.75):
        predicted_update, target_update, error_map = velocity_diagnostics(
            noisy[t], t, image, epsilon, condition, predict_velocity)
        if predicted_update.shape != image.shape or target_update.shape != image.shape:
            raise ValueError("Each update should have the same shape as the input image.")
        if error_map.shape != (image.shape[0], *image.shape[-2:]):
            raise ValueError("The error map should have shape (B, H, W); average only over RGB.")
        velocity_results[t] = tuple(value.detach().float().cpu() for value in
                                    (predicted_update, target_update, error_map))

    signed_limit = max(1e-8, *(float(value.abs().max())
                             for predicted, target, _ in velocity_results.values()
                             for value in (predicted, target)))
    error_max = ERROR_VMAX
    if error_max is None:
        error_max = max(1e-8, *(float(error.max()) for _, _, error in velocity_results.values()))
    if error_max <= ERROR_VMIN:
        raise ValueError("Choose ERROR_VMAX greater than ERROR_VMIN.")

    print("Original image")
    display(to_pil(image))
    save("velocity_original", image)
    for t, (predicted_update, target_update, error_map) in velocity_results.items():
        fig, axes = plt.subplots(1, 4, figsize=(16, 4), layout="constrained")
        fig.suptitle(f"t = {t:.2f}")
        axes[0].imshow(to_pil(noisy[t]))
        axes[1].imshow(signed_rgb(predicted_update, signed_limit))
        axes[2].imshow(signed_rgb(target_update, signed_limit))
        axes[3].imshow(error_map[0].numpy(), vmin=ERROR_VMIN, vmax=error_max)
        for ax, title in zip(axes, ("Noisy image", r"Predicted $(1-t)\hat v$",
                                    r"Ground truth $(1-t)v_{\mathrm{gt}}$", "Error (RGB RMS)")):
            ax.set_title(title)
            ax.axis("off")
        fig.savefig(OUT / f"velocity_{t:.2f}.png", dpi=110)
        plt.show()
        plt.close(fig)

## 1.5 Iterative Denoising with Euler Sampling

Implement `euler_sample(x_start,t_start,step_size,velocity)`:
xₜ₊ₕ=xₜ+h·v(xₜ,t). Recompute velocity each step and shorten the final step to reach 1 exactly.
Return the final tensor and saved states. Denoise the saved t=0.50 input with h=0.02 (25 updates).
Show the initial state, every fifth update, and final result, labeled by time. Compare blur, one-step,
and iterative results on this same input.

In [ ]:
if GROUP in ("all","core"):
    calls=[]
    def constant_v(x,t):
        calls.append(t); return torch.ones_like(x)
    toy=torch.zeros(1,3,4,4,device=device)
    for start,step,expected in [(0,.02,50),(.5,.02,25),(.93,.02,4)]:
        calls.clear(); end,states=euler_sample(toy,start,step,constant_v)
        assert len(calls)==expected and states[-1][0]==1
        assert torch.allclose(end,torch.full_like(toy,1-start),atol=1e-6)
    iterative,trajectory=euler_sample(noisy[.5],.5,H,lambda x,t:predict_velocity(x,t,condition))
    panel("trajectory",{f"t={t:.2f}":x for t,x in trajectory},3)
    panel("comparison",{"original":image,"blur":blurred[.5],"one step":one_step[.5],"Euler":iterative})
    metrics["reconstruction_mse"]={k:float(((v-image)**2).mean()) for k,v in
        {"blur":blurred[.5],"one_step":one_step[.5],"euler":iterative}.items()}
    metrics["euler_checks"]="50 full steps, 25 half steps, shortened final step passed"

## 1.6 Generating Images from Noise

Generate five independent Gaussian initial states and integrate from 0 to 1 (50 Euler updates).
Use “a high quality photo”. Show all five results and record the seeds.

In [ ]:
if GROUP in ("all","core"):
    samples={}
    for i in range(5):
        x,_=euler_sample(noise(SEED+i),0,H,lambda x,t:predict_velocity(x,t,condition))
        samples[f"seed={SEED+i}"]=x; save(f"conditional_{i}",x)
    panel("conditional_samples",samples)

# 2. Conditional Generation and Guidance

Section 1 used a fixed text prompt. Here, implement classifier-free guidance to strengthen
the prompt's effect and compare the resulting samples.

## 2.1 Classifier-Free Guidance

Implement v_cfg=v_c+w(v_c−v_u), with empty-string null condition.
Both branches must evaluate the same x and t. w=0 gives ordinary conditional prediction.
Generate five images at w=7 and h=0.02. Compare to Section 1.6 using the same initial noises.


In [ ]:
if GROUP in ("all","core"):
    x=noise(SEED)
    assert torch.equal(cfg_velocity(x,.5,condition,0,predict_velocity,null_condition),predict_velocity(x,.5,condition))
    cfg_samples={}
    for i in range(5):
        x,_=euler_sample(noise(SEED+i),0,H,guided)
        cfg_samples[f"seed={SEED+i}, w={W}"]=x; save(f"cfg_{i}",x)
    panel("cfg_samples",cfg_samples)
    metrics["cfg_zero_check"]=True

# 3. Creative Applications

Use Euler step size `h=0.02`, CFG weight `w=7`, and the empty-string prompt for the
unconditional prediction throughout this section.

## 3.1 Image-to-Image Editing

Implement `edit`: construct $(1-t_{\mathrm{start}})\epsilon+t_{\mathrm{start}}x_1$
and integrate from the starting time to 1 with CFG.
Use `"a high quality photo"` and the **seven** starting times
**0.02, 0.04, 0.06, 0.10, 0.14, 0.25, 0.50**.

Show the original and all seven edits for the **Campanile and two images of your own**.
The Campanile is included in setup. Set the two image paths below; you may reuse your image
from Section 1.1 as one of them. These inputs will also be used in Section 3.2.

In [ ]:
# The same seven times are used throughout Sections 3.1 and 3.2.
STARTS = [0.02, 0.04, 0.06, 0.10, 0.14, 0.25, 0.50]
TEST_IMAGE_PATHS = {
    "twilight_castle": "inputs_512/02_twilight_castle.jpg",
    "deep_dark_warden": "inputs_512/03_deep_dark_warden.jpg",
}

if GROUP in ("all", "editing"):
    # Load Campanile separately; `image` remains your Section 1 image.
    campanile = load_input(ROOT / "assets/campanile.jpg")
    edit_inputs = {"campanile": campanile,
                   **{name: load_input(path) for name, path in TEST_IMAGE_PATHS.items()}}
    for name, y in edit_inputs.items():
        panel("edit_" + name, {"original": y, **{
            f"t={t:.2f}": edit(y, noise(SEED), t, condition, predict_velocity, w=W, step_size=H)
            for t in STARTS}})
        print("Edited", name, flush=True)

### Editing Hand-Drawn and Web Images

Repeat the same procedure and **seven starting times** for **one image from the web and
two of your own drawings**. Download or draw the images, upload/add them to the runtime,
and set their paths below. The preprocessing helper center-crops to a square and resizes
to 512×512, so keep your drawings centered.

In [ ]:
WEB_IMAGE_PATH = "inputs_512/06_aether_sun_spirit.png"  # Path of an image you downloaded from the web.
DRAWING_PATHS = {
    "drawing_twilight_castle": "inputs_512/07_handdrawn_twilight_castle.png",
    "drawing_ice_dragon": "inputs_512/08_handdrawn_ice_dragon.png",
}

if GROUP in ("all", "editing"):
    creative_inputs = {"web_image": load_input(WEB_IMAGE_PATH),
                       **{name: load_input(path) for name, path in DRAWING_PATHS.items()}}
    for name, y in creative_inputs.items():
        panel("edit_" + name, {"original": y, **{
            f"t={t:.2f}": edit(y, noise(SEED), t, condition, predict_velocity, w=W, step_size=H)
            for t in STARTS}})
        print("Edited", name, flush=True)

## 3.2 Text-Guided Image Editing

Reuse the Campanile and two personal images from Section 3.1. For each image, choose a prompt
describing your desired edit and replace the generic photo prompt with its embedding.
Use the same **seven starting times: 0.02, 0.04, 0.06, 0.10, 0.14, 0.25, 0.50**.
Show the original and all edits, with the chosen prompt and times.

Larger starting times preserve more of the original image. Smaller starting times add more
noise and give the model more room to change the image according to the prompt.

In [ ]:
EDIT_PROMPTS = {
    "campanile": "a photograph of a rocket on a launch pad",
    "twilight_castle": "a cinematic photograph of a haunted gothic castle in a dark enchanted forest",
    "deep_dark_warden": "a cinematic photograph of a bioluminescent stone guardian in an ancient cavern",
}

if GROUP in ("all", "editing"):
    if set(EDIT_PROMPTS) != set(edit_inputs):
        raise ValueError("Provide one editing prompt for each image from Section 3.1.")
    for name,prompt in EDIT_PROMPTS.items():
        c=model.encode(prompt); y=edit_inputs[name]
        panel("text_edit_"+name,{"original":y,**{
            f"t={t:.2f}":edit(y,noise(SEED),t,c,predict_velocity,w=W,step_size=H)
            for t in STARTS}},title=f'{name}: "{prompt}"')
        print("Text edited",name,flush=True)


## 3.3 Visual Anagrams

Let F rotate an image 180 degrees (flip both spatial axes, not just vertical reflection).
Average v_cfg(x,t,c_a) and F(v_cfg(F(x),t,c_b)), and integrate using the same Euler sampler.
Choose two prompt pairs and generate two illusions. Show each upright and rotated with both prompts. Several prompt/seed attempts
may be needed: a successful numerical run does not imply both subjects are recognizable.

In [ ]:
if GROUP in ("all","illusions"):
    assert torch.equal(rotate180(rotate180(image)),image)
    # Example prompt pairs; edit these to choose your two illusions.
    pairs=[("an oil painting of an old man","an oil painting of people around a campfire"),
           ("a fantasy painting of a dark stone fortress", "a fantasy painting of a fire dragon with spread wings")]
    for i,(a,b) in enumerate(pairs):
        x=visual_anagrams(noise(SEED+i),model.encode(a),model.encode(b),predict_velocity,w=W,step_size=H)
        save(f"anagram_{i}",x); panel(f"anagram_views_{i}",{a:x,b:rotate180(x)},2)
    metrics["rotation_involution_check"]=True

## 3.4 Hybrid Images

Implement `make_hybrids` using `lowpass(v_a) + v_b - lowpass(v_b)` in each Euler update.
Prompt `a` determines the distant structure; prompt `b` determines the close-up detail.
Use the same image and time for both CFG predictions. The provided low-pass helper uses
Gaussian blur with sigma 16 and kernel size 265 at 512×512.

Choose two prompt pairs and generate **two hybrid images**. Show each at full and reduced size,
labeling both prompts. The example pairs below match the subjects and viewing directions
shown on the webpage; you can choose other pairs. Several attempts may be needed.

In [ ]:
if GROUP in ("all", "illusions"):
    # (distant / low-frequency prompt, close / high-frequency prompt)
    pairs = [("a lithograph of a skull", "a lithograph of a waterfall"),
             ("a fantasy painting of an ice dragon", "a fantasy painting of a ruined castle")]
    for i, (distant_prompt, close_prompt) in enumerate(pairs):
        x = make_hybrids(noise(SEED+i), model.encode(distant_prompt), model.encode(close_prompt),
                         predict_velocity, w=W, step_size=H)
        save(f"hybrid_{i}", x)
        print("Close:", close_prompt)
        display(to_pil(x))
        print("Distant:", distant_prompt)
        display(to_pil(x).resize((64, 64), Image.Resampling.LANCZOS))
    metrics["hybrid_lowpass_sigma"] = 16

# Bells & Whistles

**Required for CS280A students:**
- Create visual anagrams using two additional transformations from the
  [Visual Anagrams paper](https://arxiv.org/abs/2311.17919).
- Design a course logo, for example by applying text-guided image editing to a drawing
  or the UCB logo.

**Optional for everyone:** Explore your own ideas. Add cells below for your experiments
and include the results on your project webpage.

In [ ]:
# Add your Bells & Whistles experiments here.

### Save your results

Inspect your saved images as well as the numerical checks. The cell below saves the run's
settings and metrics alongside its figures. Save this notebook with its outputs before
disconnecting from the runtime.

In [ ]:
metrics["peak_memory_gib"]=torch.cuda.max_memory_allocated()/2**30
(OUT/"metrics.json").write_text(json.dumps(metrics,indent=2))
print(json.dumps(metrics,indent=2))

# Deliverable Checklist

- Include every section's required results and explanations on your project webpage.
- Submit your website PDF and code to the corresponding Gradescope assignments.
- The Google Form is not required for Part A; complete it after both parts are finished.
- CS280A students should also include both required Bells & Whistles tasks.

The project is due October 20, 2026. Follow the assignment webpage for complete submission instructions.